# 🎵 Simple AI Music Generator
This notebook trains a single basic LSTM model to learn notes from a dataset and generate a simple melody saved as a MIDI file.

## Step 1 — Install Packages and Import Libraries

In [ ]:
# Install required libraries if needed
# !pip install music21 tensorflow numpy

import urllib.request
import pickle
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from music21 import stream, note as m21note

## Step 2 — Download and Load Data

In [ ]:
# Download dataset
url = 'https://raw.githubusercontent.com/czhuang/JSB-Chorales-dataset/master/jsb-chorales-quarter.pkl'
urllib.request.urlretrieve(url, 'jsb_chorales.pkl')

with open('jsb_chorales.pkl', 'rb') as f:
    data = pickle.load(f, encoding='latin1')

train_data = data['train']
valid_data = data['valid']

print(f'Loaded {len(train_data)} training tracks.')

Loaded 229 training tracks.


## Step 3 — Data Preprocessing

In [ ]:
SEQ_LEN = 30

# Simplify by using just the first voice (melody)
def get_melody_notes(dataset):
    notes = []
    for track in dataset:
        for timestep in track:
            if len(timestep) > 0:
                notes.append(int(timestep[0])) # Just take the first note
    return notes

train_notes = get_melody_notes(train_data)
valid_notes = get_melody_notes(valid_data)

# Create vocabulary mapping mapping unique notes to integers
unique_notes = sorted(list(set(train_notes)))
note_to_int = {note: i for i, note in enumerate(unique_notes)}
int_to_note = {i: note for i, note in enumerate(unique_notes)}
vocab_size = len(unique_notes)

print(f'Total notes: {len(train_notes)} | Unique notes: {vocab_size}')

# Prepare input sequences and targets
def create_sequences(note_list):
    X, y = [], []
    for i in range(len(note_list) - SEQ_LEN):
        seq_in = note_list[i:i + SEQ_LEN]
        seq_out = note_list[i + SEQ_LEN]
        # Only use if notes exist in vocab mapping
        if all(n in note_to_int for n in seq_in) and seq_out in note_to_int:
            X.append([note_to_int[n] for n in seq_in])
            y.append(note_to_int[seq_out])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_notes)
X_valid, y_valid = create_sequences(valid_notes)

y_train_cat = to_categorical(y_train, num_classes=vocab_size)
y_valid_cat = to_categorical(y_valid, num_classes=vocab_size)

Total notes: 13789 | Unique notes: 38


## Step 4 — Build the Model

# Early Stopping

In [ ]:
checkpoint = ModelCheckpoint(
    'best_music_model.keras',
    monitor='val_loss',
    save_best_only=True
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,             # Stop if it doesn't improve for 5 epochs
    restore_best_weights=True
)

In [ ]:
model = Sequential([
    Embedding(vocab_size, 64, input_length=SEQ_LEN),
    LSTM(128, return_sequences=False),
    Dense(vocab_size, activation='softmax')
])

model.build(input_shape=(None, SEQ_LEN))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 64)         │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 106,150 (414.65 KB)

 Trainable params: 106,150 (414.65 KB)

 Non-trainable params: 0 (0.00 B)

## Step 5 — Train the Model

In [ ]:
# Short training process
model.fit(
    X_train, y_train_cat,
    validation_data=(X_valid, y_valid_cat),
    epochs=50,
    batch_size=64,
    callbacks=[checkpoint, early_stop] # Pass them here
)

Epoch 1/50
215/215 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.2239 - loss: 2.6884 - val_accuracy: 0.2688 - val_loss: 2.3866
Epoch 2/50
215/215 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.2717 - loss: 2.3372 - val_accuracy: 0.2954 - val_loss: 2.2006
Epoch 3/50
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.2902 - loss: 2.2147 - val_accuracy: 0.3011 - val_loss: 2.1525
Epoch 4/50
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.2998 - loss: 2.1636 - val_accuracy: 0.3104 - val_loss: 2.1153
Epoch 5/50
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - accuracy: 0.3133 - loss: 2.1251 - val_accuracy: 0.3148 - val_loss: 2.1018
Epoch 6/50
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.3247 - loss: 2.0902 - val_accuracy: 0.3247 - val_loss: 2.0859
Epoch 7/50
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.3348 - loss: 2.0561 - val_accuracy: 0.3324 - val_loss: 2.0688
Epoch 8/50
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3481 - loss: 2.0246 - val_accura

## Step 6 — Generate Music

In [ ]:
# Pick a random seed from data to start generation
seed_index = np.random.randint(0, len(X_valid) - 1)
current_sequence = list(X_valid[seed_index])

generated_notes = []

# Predict 100 notes sequentially
for _ in range(100):
    input_data = np.array([current_sequence[-SEQ_LEN:]])
    prediction = model.predict(input_data, verbose=0)[0]

    # Choose note based on probability distributions
    predicted_idx = np.random.choice(len(prediction), p=prediction)

    generated_notes.append(int_to_note[predicted_idx])
    current_sequence.append(predicted_idx)

print(f'Generated {len(generated_notes)} notes.')

Generated 100 notes.


## Step 7 — Export Output to MIDI

In [ ]:
output_score = stream.Score()
output_part = stream.Part()

for mid_note in generated_notes:
    n = m21note.Note(mid_note)
    n.quarterLength = 1.0  # Keep a single standard duration for every note
    output_part.append(n)

output_score.append(output_part)
output_score.write('midi', fp='simple_output.mid')
print('Saved result to simple_output.mid')

Saved result to simple_output.mid
